<a href="https://colab.research.google.com/github/ximecaceres/materia-IA-/blob/main/ventasoficial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### SECCIÓN 1: CARGAR EL DATASET E IMPORTAR LIBRERÍAS
En esta parte inicial, prepararemos el entorno de trabajo instalando y llamando a las herramientas necesarias.

In [1]:
import pandas as pd

try:
    import langchain
    print("Librería LangChain importada correctamente")
except ImportError:
    # Si no está instalada, mostramos cómo hacerlo
    print("Librería LangChain no detectada. Instalando...")
    !pip install langchain -q
    import langchain
    print("Librería LangChain importada correctamente")

print("Librería Pandas importada correctamente")

Librería LangChain importada correctamente
Librería Pandas importada correctamente


In [2]:
# Cargamos el archivo CSV que ya tienes en el entorno de Colab
# Usamos la función read_csv de pandas con una codificación estándar
try:
    df = pd.read_csv('/content/sales_data_sample.csv', encoding='unicode_escape')
    print("Archivo 'sales_data_sample.csv' cargado exitosamente en la variable 'df'")
except Exception as e:
    print(f"Error al cargar el archivo: {e}")

Archivo 'sales_data_sample.csv' cargado exitosamente en la variable 'df'


### SECCIÓN 2: NORMALIZACIÓN Y LIMPIEZA PASO A PASO
Ahora transformaremos los datos para que sean fáciles de procesar por cualquier modelo de IA.

In [3]:
# 1. Normalizar nombres de columnas
# Convertimos todo a minúsculas y reemplazamos espacios por guiones bajos
df.columns = [col.lower().replace(' ', '_') for col in df.columns]

print("Nombres de columnas normalizados:")
print(df.columns.tolist())

Nombres de columnas normalizados:
['ordernumber', 'quantityordered', 'priceeach', 'orderlinenumber', 'sales', 'orderdate', 'status', 'qtr_id', 'month_id', 'year_id', 'productline', 'msrp', 'productcode', 'customername', 'phone', 'addressline1', 'addressline2', 'city', 'state', 'postalcode', 'country', 'territory', 'contactlastname', 'contactfirstname', 'dealsize']


In [4]:
# 2. Verificar valores nulos
# Mostramos cuántos datos faltan en cada columna
print("Valores nulos por columna antes de la limpieza:")
display(df.isnull().sum())

Valores nulos por columna antes de la limpieza:


,0
ordernumber,0
quantityordered,0
priceeach,0
orderlinenumber,0
sales,0
orderdate,0
status,0
qtr_id,0
month_id,0
year_id,0


In [5]:
# 3. Reemplazar valores nulos de forma didáctica

# Para columnas numéricas: llenamos con 0
numeric_cols = df.select_dtypes(include=['number']).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

# Para columnas de texto (objetos): llenamos con 'No especificado'
text_cols = df.select_dtypes(include=['object']).columns
df[text_cols] = df[text_cols].fillna('No especificado')

print("Limpieza de nulos completada: los números vacíos ahora son 0 y los textos vacíos son 'No especificado'.")

Limpieza de nulos completada: los números vacíos ahora son 0 y los textos vacíos son 'No especificado'.


In [6]:
# 4. Limpieza de caracteres especiales en columnas de texto
# Definimos una función simple para limpiar el texto
def limpiar_texto(texto):
    if isinstance(texto, str):
        # Ejemplo simple: quitar comillas dobles y caracteres raros comunes
        texto = texto.replace('"', '').replace("'", "")
        # Aquí podrías añadir más reemplazos según necesites
        return texto
    return texto

# Aplicamos la limpieza a todas las columnas de texto
for col in text_cols:
    df[col] = df[col].apply(limpiar_texto)

print("Caracteres especiales básicos han sido limpiados de las columnas de texto.")

Caracteres especiales básicos han sido limpiados de las columnas de texto.


In [7]:
# 5. Mostrar estructura final y estadísticas
print("--- Información del Dataset ---")
df.info()

print("\n--- Estadísticas Generales ---")
display(df.describe())

--- Información del Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ordernumber       2823 non-null   int64  
 1   quantityordered   2823 non-null   int64  
 2   priceeach         2823 non-null   float64
 3   orderlinenumber   2823 non-null   int64  
 4   sales             2823 non-null   float64
 5   orderdate         2823 non-null   object 
 6   status            2823 non-null   object 
 7   qtr_id            2823 non-null   int64  
 8   month_id          2823 non-null   int64  
 9   year_id           2823 non-null   int64  
 10  productline       2823 non-null   object 
 11  msrp              2823 non-null   int64  
 12  productcode       2823 non-null   object 
 13  customername      2823 non-null   object 
 14  phone             2823 non-null   object 
 15  addressline1      2823 non-null   object 
 16  addresslin

,ordernumber,quantityordered,priceeach,orderlinenumber,sales,qtr_id,month_id,year_id,msrp
count,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.00000,2823.000000
mean,10258.725115,35.092809,83.658544,6.466171,3553.889072,2.717676,7.092455,2003.81509,100.715551
std,92.085478,9.741443,20.174277,4.225841,1841.865106,1.203878,3.656633,0.69967,40.187912
min,10100.000000,6.000000,26.880000,1.000000,482.130000,1.000000,1.000000,2003.00000,33.000000
25%,10180.000000,27.000000,68.860000,3.000000,2203.430000,2.000000,4.000000,2003.00000,68.000000
50%,10262.000000,35.000000,95.700000,6.000000,3184.800000,3.000000,8.000000,2004.00000,99.000000
75%,10333.500000,43.000000,100.000000,9.000000,4508.000000,4.000000,11.000000,2004.00000,124.000000
max,10425.000000,97.000000,100.000000,18.000000,14082.800000,4.000000,12.000000,2005.00000,214.000000


### SECCIÓN 3: GUARDAR EL RESULTADO
Finalmente, guardamos nuestro trabajo en un nuevo archivo limpio.

In [9]:
# Exportamos el DataFrame a un nuevo archivo CSV
df.to_csv('ventasxime.csv', index=False)

print("¡El dataset ha sido normalizado y guardado con éxito como sales_data_cleaned.csv!")

¡El dataset ha sido normalizado y guardado con éxito como sales_data_cleaned.csv!


### VERIFICACIÓN DEL ARCHIVO GENERADO
En esta celda cargaremos el nuevo archivo para comprobar que la limpieza se guardó correctamente.

In [10]:
# 1. Cargamos el nuevo archivo generado
df_verificacion = pd.read_csv('ventasxime.csv')

# 2. Comprobamos si hay nulos (debería salir todo en 0)
print("Conteo de valores nulos en el nuevo archivo:")
print(df_verificacion.isnull().sum().sum())

# 3. Mostramos las primeras filas para validar visualmente la limpieza
print("\nVista previa de los datos normalizados y limpios:")
display(df_verificacion.head())

Conteo de valores nulos en el nuevo archivo:
0

Vista previa de los datos normalizados y limpios:


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,No especificado,NYC,NY,10022,USA,No especificado,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de lAbbaye,No especificado,Reims,No especificado,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,No especificado,Paris,No especificado,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,No especificado,Pasadena,CA,90003,USA,No especificado,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,No especificado,San Francisco,CA,No especificado,USA,No especificado,Brown,Julie,Medium


### CONFIGURACIÓN DEL AGENTE CON CONTEXTO DETALLADO
En este bloque inicializamos el modelo de lenguaje y configuramos el agente con el diccionario de columnas para asegurar la máxima precisión en sus respuestas.

In [19]:
# 1. Instalación de librerías necesarias de forma silenciosa
!pip install langchain langchain-mistralai langchain-experimental pandas -q

import pandas as pd
from google.colab import userdata
from langchain_mistralai import ChatMistralAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# 2. Recuperación de la clave API desde los secretos de Colab
try:
    mistral_key = userdata.get('MISTRAL_API_KEY')
except Exception as e:
    print("❌ ERROR: No se pudo acceder a MISTRAL_API_KEY.")
    print("Por favor, ve al icono de la llave (Secrets) a la izquierda y activa el 'Notebook access' para MISTRAL_API_KEY.")
    mistral_key = None

if mistral_key:
    # 3. Inicialización del LLM (Mistral) con temperatura 0
    llm = ChatMistralAI(api_key=mistral_key, model="mistral-small-latest", temperature=0)

    # 4. Carga de los datos normalizados
    df = pd.read_csv('ventasxime.csv')

    # 5. Definición del Prompt con el diccionario de datos según la cátedra
    prefix_prompt = """
    Sos un analista de datos experto en ventas.
    Trabajás con un DataFrame llamado 'df' que tiene estas columnas:
    - ordernumber: número de orden
    - quantityordered: cantidad de productos pedidos
    - priceeach: precio unitario
    - sales: total de la venta
    - orderdate: fecha del pedido
    - status: estado del pedido (Shipped, Cancelled, Resolved, etc.)
    - qtr_id: id del trimestre
    - month_id: número de mes
    - year_id: año del pedido
    - productline: línea de producto (Motorcycles, Classic Cars, etc.)
    - customername: nombre del cliente
    - city: ciudad del cliente
    - country: país del cliente
    - dealsize: tamaño del deal (Small, Medium, Large)

    Instrucciones:
    1. Para resolver cualquier pregunta numérica, estadística o de filtros (como sumar ventas, promedios o buscar el año con mayores ventas), DEBES escribir y ejecutar código de Python con Pandas sobre el DataFrame 'df'.
    2. Responde siempre en español, de forma clara y concisa.
    3. Si no podés responder en base a los datos, decilo claramente.
    """

    # 6. Creación del agente de Pandas
    agent = create_pandas_dataframe_agent(
        llm,
        df,
        prefix=prefix_prompt,
        verbose=True,
        allow_dangerous_code=True,
        agent_type="tool-calling"
    )

    # 7. Confirmación final
    print("✓ Agente configurado con el contexto detallado del dataset.")

✓ Agente configurado con el contexto detallado del dataset.


### PRUEBA DE CONSULTA DE NEGOCIO
Utilizamos el método invoke para que el agente analice los datos y determine el año con mejor desempeño comercial.

In [20]:
# Definimos la consulta específica
query = "¿Cuál fue el año que mayor ventas (sales) tuvo?"

# Ejecutamos el agente y guardamos el resultado
respuesta = agent.invoke(query)

# Mostramos la conclusión final
print(f"\nResultado del análisis: {respuesta['output']}")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df.groupby('year_id')['sales'].sum().idxmax()"}`


2004El año con mayores ventas (sales) fue **2004**.

> Finished chain.

Resultado del análisis: El año con mayores ventas (sales) fue **2004**.


In [22]:
# Segunda prueba de consultas
# Definimos la consulta específica
query = "¿Cuales son los 3 países que generaron más ingresos totales?"

# Ejecutamos el agente y guardamos el resultado
respuesta = agent.invoke(query)

# Mostramos la conclusión final
print(f"\nResultado del análisis: {respuesta['output']}")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Agrupar por país y sumar las ventas\ntop_countries = df.groupby('country')['sales'].sum().reset_index()\n\n# Ordenar de mayor a menor y seleccionar los top 3\ntop_3_countries = top_countries.sort_values(by='sales', ascending=False).head(3)\n\ntop_3_countries"}`


   country       sales
18     USA  3627982.83
14   Spain  1215686.92
6   France  1110916.52Los 3 países que generaron más ingresos totales son:

1. **USA**: 3,627,982.83
2. **Spain**: 1,215,686.92
3. **France**: 1,110,916.52

> Finished chain.

Resultado del análisis: Los 3 países que generaron más ingresos totales son:

1. **USA**: 3,627,982.83
2. **Spain**: 1,215,686.92
3. **France**: 1,110,916.52


# CREACIÓN RAG

In [1]:
# 1. Instalación de librerías y reinicio automático para aplicar cambios de versión
import os

try:
    import opentelemetry.sdk.environment_variables
    from langchain_huggingface import HuggingFaceEmbeddings
    print("✓ Librerías verificadas y listas para usar.")
except (ImportError, ModuleNotFoundError):
    print("⚠ Instalando dependencias faltantes...")
    !pip install -U "opentelemetry-api>=1.24.0" "opentelemetry-sdk>=1.24.0" -q
    !pip install langchain langchain-mistralai langchain-chroma langchain-huggingface sentence-transformers chromadb pandas -q

    print("\n¡REINICIANDO ENTORNO PARA APLICAR CAMBIOS! Por favor, ejecuta esta celda una vez más.")
    os.kill(os.getpid(), 9)

import pandas as pd
from google.colab import userdata
from langchain_mistralai import ChatMistralAI
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

✓ Librerías verificadas y listas para usar.


In [2]:
# 2. PREPARACIÓN DE DOCUMENTOS (CORPUS)
# Cargamos el dataset normalizado
df_rag = pd.read_csv('ventasxime.csv')

# Función para transformar una fila en un texto descriptivo (narrativa)
def fila_a_texto(row):
    return (
        f"Orden: {row['ordernumber']}, Producto: {row['productline']}, "
        f"Cliente: {row['customername']}, Ventas: {row['sales']}, "
        f"Cantidad: {row['quantityordered']}, Precio Unitario: {row['priceeach']}, "
        f"Fecha: {row['orderdate']}, Estado: {row['status']}, "
        f"Ciudad: {row['city']}, Pais: {row['country']}, Tamaño: {row['dealsize']}."
    )

# Creamos la lista de objetos 'Document' de LangChain
documentos = [
    Document(page_content=fila_a_texto(row), metadata={"id": row['ordernumber']})
    for _, row in df_rag.iterrows()
]

print(f"✓ Se han procesado {len(documentos)} documentos para el corpus.")

✓ Se han procesado 2823 documentos para el corpus.


In [3]:
# 3. BASE VECTORIAL CON CHROMADB
# Inicializamos el modelo de embeddings (gratuito y local)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Creamos la base de datos vectorial en memoria
vectorstore = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings,
    collection_name="ventas_xime"
)

# Configuramos el retriever para buscar los 5 documentos más parecidos (k=5)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("✓ Base vectorial ChromaDB creada correctamente.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Base vectorial ChromaDB creada correctamente.


In [4]:
# 4. CONFIGURACIÓN DE LA CADENA RAG CON MISTRAL
# Recuperamos la clave API
mistral_key = userdata.get('MISTRAL_API_KEY')

# Inicializamos el LLM
llm_rag = ChatMistralAI(api_key=mistral_key, model="mistral-small-latest", temperature=0)

# 5. SYSTEM PROMPT DEL RAG
template = """
Sos un analista de datos experto en ventas y asistente de soporte RAG.
Tu tarea es responder las preguntas del usuario basándote ÚNICAMENTE en el contexto de los documentos recuperados que se te proporcionan.

Contexto de las columnas principales (en minúsculas):
ordernumber, quantityordered, priceeach, sales, orderdate, status, qtr_id, month_id, year_id, productline, customername, city, country, dealsize.

Instrucciones críticas:
- Responde siempre en español de forma clara, concisa y profesional.
- Basa tu respuesta estrictamente en los documentos del contexto.
- Si la respuesta no se puede deducir del contexto, di claramente: 'No cuento con información suficiente en el corpus para responder a esa pregunta'.

Contexto recuperado:
{context}

Pregunta del usuario:
{question}
"""

prompt_rag = ChatPromptTemplate.from_template(template)

# Construcción de la cadena usando LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_rag
    | llm_rag
    | StrOutputParser()
)

print("✓ Cadena RAG configurada y lista.")

✓ Cadena RAG configurada y lista.


In [ ]:
# 6. INTERFAZ DE CHAT INTERACTIVO
print("--- Chat de Soporte de Ventas RAG Activo ---")
print("(Escribe 'salir' para finalizar)\n")

while True:
    usuario_input = input("Tu pregunta sobre ventas: ")

    if usuario_input.lower() == 'salir':
        print("Finalizando sesión de chat. ¡Hasta luego!")
        break

    if usuario_input.strip() == "":
        continue

    # Procesar la consulta a través de la cadena RAG
    try:
        respuesta_rag = rag_chain.invoke(usuario_input)
        print(f"\nAsistente RAG: {respuesta_rag}")
    except Exception as e:
        print(f"\nOcurrió un error al procesar la consulta: {e}")

    print("\n" + "-"*30)
    print("¿Quieres preguntarme algo más?")

--- Chat de Soporte de Ventas RAG Activo ---
(Escribe 'salir' para finalizar)

Tu pregunta sobre ventas: ¿Qué ventas se registraron en la ciudad de Madrid?

Asistente RAG: Las ventas registradas en la ciudad de Madrid son las siguientes:

- **Orden 10424**:
  - Ventas: 2702.04 (Small)
  - Ventas: 7969.36 (Large)
  - Ventas: 7182.0 (Large)

- **Orden 10412**:
  - Ventas: 925.3 (Small)
  - Ventas: 8498.0 (Large)

**Total de ventas en Madrid**: 27,276.70

------------------------------
¿Quieres preguntarme algo más?
Tu pregunta sobre ventas: ¿Tuvimos operaciones comerciales en la capital de España?

Asistente RAG: Sí, tuvimos operaciones comerciales en Madrid (capital de España). Según los registros, se realizaron ventas a clientes como **Euro Shopping Channel** y **Iberia Gift Imports, Corp.** en esta ciudad, con pedidos como los siguientes:

- **Orden 10424**: 3 transacciones diferentes (ventas totales: **17,853.40**).
- **Orden 10412**: 1 transacción (ventas: **925.30**).

Todos los pe